# Computational Methods — Web Scraping
## Getting started with web scraping

In this class, we scrape a small practice website to learn the basics of getting data from the web with Python.

We will learn how to:

* read a URL and the basic structure of HTML;
* download a web page safely (with a timeout, saved to disk);
* parse a page with BeautifulSoup;
* find tags, attributes, and text;
* search with `find_all()` and CSS selectors;
* collect the results into a table and save it.

### Before you start

* **Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.
* **The page loads itself.** There is nothing to download or upload. The cells below download the practice page (quotes.toscrape.com) for you; you just need to be online. If the connection is slow, a backup copy of the page is built in.

# Web Scraping

How to get data from the web for free (* legal caveats apply)!

We will scrape [quotes.toscrape.com](https://quotes.toscrape.com), a small practice website that was built for learning scraping, so it is safe to use.

Based on tutorials at [dataquest.io](https://www.dataquest.io/blog/web-scraping-tutorial-python/) and [realpython.com](https://realpython.com/beautiful-soup-web-scraper-python/)

# Legal matters

Is it legal to scrape data? It depends. Some websites explicitly allow web scraping, others explicitly forbid it, and many give no clear guidance.

Before scraping any website, look for its **terms and conditions** (and its `robots.txt`) to see if there are rules about scraping. If there are, follow them. If there are not, it is your judgement call.

Scraping uses the website's server resources. One page is no problem, but thousands of pages can become expensive for the site owner. So:

* do not scrape more often than you need to
* **save** every page you download and work from the saved copy while you develop your code
* pause between requests with `time.sleep()` so you do not overwhelm the server
* always set a **timeout**, so your code does not wait forever if the server does not answer

# URLs

Let's start with the address, the URL. It already contains useful information. Example:

`https://example.com/search/?q=Software-Developer&where=Australia`

A URL has two main parts:

1. The **base URL**, `https://example.com/search/`, the path to a page or function on the server
2. The **query parameters**, `?q=Software-Developer&where=Australia`. They always start with `?` and are `key=value` pairs, separated by an ampersand (`&`).

Changing the query parameters changes the page you get back, and vice versa. If you know the keys, you can build the URL yourself instead of typing it into a search box.

Our practice site works the same way, just with the information inside the path:

* `https://quotes.toscrape.com/` is page 1
* `https://quotes.toscrape.com/page/2/` is page 2
* `https://quotes.toscrape.com/tag/love/` shows only quotes tagged "love"

Exploring the URLs in your browser first helps you download only what you really need.

# HTML

HyperText Markup Language (HTML) is not a programming language like Python, but a **markup language** that tells the browser how to display content. It is similar to marking text as bold or as a title in Word.

HTML consists of **tags** in angle brackets `<` `>`. Tags usually come in pairs: an **opening tag** and a **closing tag** that repeats the name with a slash `/`. The content sits in between.

```html
<html>
    This is interpreted as HTML
</html>
```

Indenting is optional (unlike Python), but makes HTML easier to read.

Tags can be **nested**. An `<html>` tag contains a `<head>` (page title, links to style files, meta-data) and a `<body>` (the content we usually want):

```html
<html>
    <head>
        <title>This title shows in your tab</title>
    </head>

    <body>
        This is interpreted as HTML
    </body>
</html>
```

Related tags have family names:

1. A **child** is a tag inside another tag. `<head>` and `<body>` are children of `<html>`.
2. A **parent** is the outer tag. `<html>` is the parent of `<head>` and `<body>`.
3. **Siblings** share the same parent. `<head>` and `<body>` are siblings.

### Common tags

* `<p>` paragraph
* `<h1>` to `<h6>` headings of different sizes
* `<div>` a block (section) of the page
* `<span>` marks a few words inside a line
* `<b>` bold, `<i>` or `<em>` italic
* `<table>` table, with rows (`<tr>`), headers (`<th>`) and cells (`<td>`)
* `<a>` (anchor) a link to another page

### Attributes

Tags can have **attributes** written as `key="value"`. For example, a link stores its target in `href`:

```html
Here is a <a href="https://example.com">website</a>.
```

Two special attributes are `class` and `id`. They give elements names, so that CSS (and our scraper) can find them.

* An element can have several classes, and a class can be used by many elements.
* An `id` is used only once per page.

```html
<html>
    <head>
        <title>This title shows in your tab</title>
    </head>

    <body>
        <p class="intro">
            <span id="lead">This is interpreted as HTML.</span>
            Here is a <a href="https://example.com">website</a>.
        </p>
    </body>
</html>
```

### Activity

Copy the code above into a text editor (NOT Word) and save it as `test.html`. Open the file in your browser. Then try some of the other tags and attributes.

### Aside: CSS

Styles (colors, fonts, layout) live in separate CSS files and refer to tags, classes and ids. CSS does not change the content, so we mostly ignore it. But later we will reuse its **selector** syntax to find elements.

# Download a page

To scrape a page we first download it, using the `requests` library. A few things make downloading safe:

* `timeout=10`: give up after 10 seconds instead of waiting forever
* `headers`: tell the server who we are
* **saving to a file**: next time we use the saved copy and do not hit the server again
* `time.sleep(1)`: a short pause after each download
* `try/except`: if the download fails, the notebook keeps going instead of crashing

Run the next cell. You do not need to understand every line yet.

In [ ]:
import os
import time
import requests
from bs4 import BeautifulSoup

def get_page(url, filename):
    """Return the HTML of a page as text. Uses the saved copy if it exists.
    Returns None if the download fails."""
    # 1. already saved? then do not download again
    if os.path.exists(filename):
        print("Using saved copy:", filename)
        with open(filename, encoding="utf8") as f:
            return f.read()

    # 2. otherwise download (and never wait longer than 10 seconds)
    try:
        response = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0 (class exercise)"})
        response.raise_for_status()   # raises an error for 404, 500, ...
    except requests.RequestException as error:
        print("Download failed:", error)
        return None

    # 3. save it and pause politely
    response.encoding = "utf-8"
    with open(filename, "w", encoding="utf8") as f:
        f.write(response.text)
    print("Downloaded", url, "- status", response.status_code)
    time.sleep(1)
    return response.text

The **status code** tells us if the download worked: `200` means success, codes starting with `4` or `5` are errors (`404` is the infamous "Page not found").

Now let's download the first page:

In [ ]:
url = "https://quotes.toscrape.com/"
page = get_page(url, "quotes_page1.html")

**If the download failed** (no internet, or the site is slow), run the next cell. It holds a small backup copy of the page, with the same structure, so you can follow the rest of the class. If the download worked, it does nothing.

In [ ]:
BACKUP_HTML = """
<html lang="en">
<head><title>Quotes to Scrape</title></head>
<body>
<div class="container">
  <div class="row header-box">
    <h1><a href="/">Quotes to Scrape</a></h1>
    <p><a href="/login">Login</a></p>
  </div>
  <div class="quote">
    <span class="text">“It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.”</span>
    <span>by <small class="author">Jane Austen</small> <a href="/author/Jane-Austen">(about)</a></span>
    <div class="tags">Tags:
      <a class="tag" href="/tag/aliteracy/page/1/">aliteracy</a>
      <a class="tag" href="/tag/books/page/1/">books</a>
      <a class="tag" href="/tag/classic/page/1/">classic</a>
      <a class="tag" href="/tag/humor/page/1/">humor</a>
    </div>
  </div>
  <div class="quote">
    <span class="text">“The world as we have created it is a process of our thinking.”</span>
    <span>by <small class="author">Albert Einstein</small> <a href="/author/Albert-Einstein">(about)</a></span>
    <div class="tags">Tags:
      <a class="tag" href="/tag/change/page/1/">change</a>
      <a class="tag" href="/tag/deep-thoughts/page/1/">deep-thoughts</a>
      <a class="tag" href="/tag/thinking/page/1/">thinking</a>
      <a class="tag" href="/tag/world/page/1/">world</a>
    </div>
  </div>
  <div class="quote">
    <span class="text">“Imperfection is beauty, madness is genius.”</span>
    <span>by <small class="author">Marilyn Monroe</small> <a href="/author/Marilyn-Monroe">(about)</a></span>
    <div class="tags">Tags:
      <a class="tag" href="/tag/be-yourself/page/1/">be-yourself</a>
      <a class="tag" href="/tag/inspirational/page/1/">inspirational</a>
    </div>
  </div>
  <nav><ul class="pager"><li class="next"><a href="/page/2/">Next</a></li></ul></nav>
</div>
</body>
</html>
"""

if page is None:
    page = BACKUP_HTML
    print("Using the backup copy of the page.")

## Activity

Choose a page you are allowed to scrape (a public page, e.g. a weather service) and download it with `get_page(url, "my_page.html")`. Check that you got text back, not `None`.

In [ ]:
# Your code here

# BeautifulSoup

The `page` we downloaded is one long string. The `BeautifulSoup` library **parses** it, i.e., turns it into a structure we can search. We create a `BeautifulSoup` object and say which parser to use (`html.parser`).

In [ ]:
soup = BeautifulSoup(page, "html.parser")

`prettify()` prints the HTML with nice indentation. Pages can be very long, so we only print the first 1000 characters:

In [ ]:
print(soup.prettify()[:1000])

## Navigating the tags

Since tags are nested, we can reach them like attributes: `soup.title` is the `<title>` tag, `soup.body` is the `<body>` tag.

> **Tip:** always look at the structure of the page first (in your browser: right-click, "Inspect"). Not all pages are the same, so you will have to adapt your code to each page.

In [ ]:
print(soup.title)
print(soup.title.get_text())

The `get_text()` method removes all tags and gives us just the text of a tag (and everything inside it):

In [ ]:
body = soup.body
text = body.get_text()
print(text[:300])

The text still has line breaks (`\n`) and extra spaces. `get_text()` can clean this up for us: `strip=True` removes whitespace around each piece, and `separator=" "` joins the pieces with a space.

In [ ]:
clean_text = body.get_text(separator=" ", strip=True)
print(clean_text[:300])

## Activity

Extract the text from the body of the page you downloaded in the previous activity. Print only the first 500 characters.

In [ ]:
# Your code here

# Finding *all* instances of a tag

Walking through the children takes many lines. If we want a specific tag, `find_all()` returns a **list** of all matches:

In [ ]:
links = soup.find_all("a")
print(len(links))
print(links[0].prettify())

Because `find_all()` returns a list, we loop over it to get the texts:

In [ ]:
for link in links[:5]:
    print(link.get_text(strip=True))

If we only need the **first** match, `find()` returns a single tag:

In [ ]:
soup.find("a")

To get the URL of a link, we need the value of its `href` attribute. A tag works like a dictionary, so we use square brackets (`.attrs` shows all attributes):

In [ ]:
first_link = soup.find("a")
print(first_link.attrs)
print(first_link["href"])

## Aside: Crawling

Scraping a whole website (if we are allowed to!) is called **crawling**. The idea:

1. extract all links on the page with `find_all("a")`
2. keep only links that belong to the website, and that we have not visited yet
3. visit the next one, and repeat

You must keep track of what you already downloaded (a `set` of URLs), so you do not loop forever. And save pages to disk, so you do not start from scratch if your code stops.

# Searching with constraints

`find_all()` can also filter by the **text** of a tag, or by its `class` or `id` (we write `class_` because `class` is a reserved word in Python):

In [ ]:
soup.find_all("a", string="Login")

In [ ]:
soup.find_all("span", class_="text")

In [ ]:
soup.find_all(class_="tag")[:5]

# Nested search with CSS selectors

We can also use **CSS selectors**, which describe tags by their position and name:

* `p a` finds all `<a>` tags inside a `<p>` tag
* `div p a` finds `<a>` inside `<p>` inside `<div>`
* `div.quote` finds `<div>` tags with the class `quote`
* `p#first` finds the `<p>` tag with the id `first`
* `div.quote a` finds `<a>` tags inside a `<div class="quote">`

The `select()` method takes a selector and returns a list, like `find_all()`:

In [ ]:
quotes = soup.select("div.quote")
print(len(quotes))
quotes[0]

A selector can also be used on a single tag. Let's look inside the first quote:

In [ ]:
first = quotes[0]
print(first.select_one("span.text").get_text(strip=True))
print(first.select_one("small.author").get_text(strip=True))
print([tag.get_text(strip=True) for tag in first.select("a.tag")])

`select_one()` returns the first match (like `find()`), `select()` returns all matches (like `find_all()`).

# Putting it together: a table of quotes

We loop over all quotes, collect the fields in a dictionary, and turn the list of dictionaries into a table with `pandas`.

In [ ]:
import pandas as pd

rows = []
for quote in soup.select("div.quote"):
    rows.append({
        "text": quote.select_one("span.text").get_text(strip=True),
        "author": quote.select_one("small.author").get_text(strip=True),
        "tags": ", ".join(tag.get_text(strip=True) for tag in quote.select("a.tag")),
    })

df = pd.DataFrame(rows)
df

In [ ]:
df.to_csv("quotes.csv", index=False)

## Activity

1. Print only the **authors** of all quotes on the page.
2. Count how many quotes have the tag `books` or `inspirational`.
3. **Bonus:** the page has a "Next" link. Find it (`li.next a`) and print its `href`.

In [ ]:
# Your code here

## Bonus: several pages

To scrape more than one page, we loop over the page URLs. Because `get_page()` saves every page and pauses after each download, this is safe to run. We still limit the loop to **3 pages**.

In [ ]:
all_rows = []
for number in range(1, 4):
    page_html = get_page(f"https://quotes.toscrape.com/page/{number}/", f"quotes_page{number}.html")
    if page_html is None:        # download failed: stop instead of waiting
        print("Stopping at page", number)
        break
    page_soup = BeautifulSoup(page_html, "html.parser")
    for quote in page_soup.select("div.quote"):
        all_rows.append({
            "page": number,
            "text": quote.select_one("span.text").get_text(strip=True),
            "author": quote.select_one("small.author").get_text(strip=True),
        })

print(len(all_rows), "quotes collected")
pd.DataFrame(all_rows).head()